## 🎯 Learning Objectives
* Understand the critical importance of latency and cost profiling in production-grade Agentic RAG systems.
* Learn how to measure and interpret key performance indicators (KPIs) such as end-to-end latency, time-to-first-token, and token-based costs.
* Implement basic profiling techniques within a simulated agentic workflow to track execution time and resource consumption.
* Identify common performance bottlenecks and cost drivers in multi-agent architectures.
* Explore strategies and tools for optimizing agentic system performance and managing operational expenses.


## Latency and Cost Profiling: Optimizing Your Agentic RAG System

In the rapidly evolving landscape of Agentic AI, building a functional system is just the first step. For any production-ready application, especially one handling real-world e-commerce queries, **performance** and **cost-efficiency** are paramount. This lesson delves into the critical practices of latency and cost profiling, equipping you with the knowledge to build not just smart, but also fast and economical agentic RAG solutions.

### Why Profile Latency and Cost?

Imagine an e-commerce customer using your Agentic RAG assistant to find a product. If the assistant takes 10-15 seconds to respond, or if each query costs several dollars, your system is unlikely to succeed. Profiling helps us understand and optimize these crucial aspects:

1.  **User Experience (Latency)**: Slow responses lead to frustrated users and abandoned sessions. Latency profiling helps identify bottlenecks in the agent's thought process, tool execution, and RAG pipeline, ensuring a snappy, responsive experience.
2.  **Operational Costs (Cost)**: Large Language Models (LLMs) and specialized APIs (like embedding models, vector databases, or external tools) incur costs per token, per call, or per compute hour. Without cost profiling, your operational expenses can quickly spiral out of control, making your solution economically unviable.
3.  **Resource Optimization**: Understanding where time and money are spent allows for informed decisions on resource allocation, model selection, caching strategies, and agent orchestration.
4.  **Scalability and Reliability**: Profiling data is essential for capacity planning. Knowing how long a typical query takes and how much it costs helps predict infrastructure needs and ensure the system remains stable under load.

### The Agentic Challenge

Profiling agentic systems is more complex than traditional applications due to their dynamic, non-deterministic nature:

*   **Multiple LLM Calls**: An agent might make several LLM calls for planning, reasoning, tool selection, and response generation.
*   **Tool Usage**: Each tool call (e.g., searching a vector database, calling an external API, running a code interpreter) adds its own latency and potential cost.
*   **Dynamic Execution Paths**: The sequence and number of steps an agent takes can vary significantly based on the query, making average profiling challenging.
*   **Context Window Management**: Managing the LLM's context window (input tokens) directly impacts both cost and latency.

### Key Metrics to Track

#### Latency:
*   **End-to-End Query Time**: Total time from user input to final agent response.
*   **Time-to-First-Token (TTFT)**: How quickly the agent starts generating its response. Crucial for perceived responsiveness.
*   **LLM Call Latency**: Time taken for each individual LLM API call.
*   **Tool Execution Latency**: Time taken for each tool to complete its task.
*   **RAG Pipeline Latency**: Time spent on retrieval, re-ranking, and context assembly.

#### Cost:
*   **LLM Token Costs**: Input tokens and output tokens for each LLM call (often priced differently).
*   **Embedding Model Costs**: Cost per embedding call or per token embedded.
*   **Vector Database Costs**: Storage, indexing, and query costs.
*   **External API Costs**: Any third-party services used by tools (e.g., product catalog APIs, payment gateways).
*   **Compute Costs**: CPU/GPU hours for running agents, RAG components, or custom tools.

### Analogies

Think of profiling an agentic system like optimizing a complex manufacturing process:

*   **Latency is like the assembly line speed**: How quickly does a raw material (query) become a finished product (answer)? Each workstation (LLM call, tool use) has its own processing time. A bottleneck at any station slows down the entire line.
*   **Cost is like the raw material and labor expenses**: Each component (token) and each worker's time (compute, API call) adds to the final product's cost. You want to ensure you're not overpaying for materials or inefficient labor.

By systematically measuring and analyzing these metrics, we can pinpoint inefficiencies and make data-driven decisions to enhance both the performance and economic viability of our Agentic RAG solutions.


In [ ]:
import time
import tiktoken
import random
from collections import defaultdict

# --- Configuration and Mock Services ---

# Hypothetical LLM costs per 1000 tokens (as of 2026, assuming tiered pricing)
LLM_COST_PER_1K_INPUT_TOKENS = 0.0015 # e.g., GPT-4o mini input
LLM_COST_PER_1K_OUTPUT_TOKENS = 0.0045 # e.g., GPT-4o mini output
EMBEDDING_COST_PER_1K_TOKENS = 0.00005 # e.g., OpenAI text-embedding-3-small

# Initialize tokenizer for token counting
tokenizer = tiktoken.get_encoding("cl100k_base")

class MockLLM:
    """Simulates an LLM API call with variable latency and token counts."""
    def __init__(self, model_name="mock-llm-2026"): 
        self.model_name = model_name

    def generate(self, prompt: str, max_tokens: int = 100) -> str:
        input_tokens = len(tokenizer.encode(prompt))
        # Simulate LLM processing time based on input tokens
        time.sleep(0.05 + input_tokens * 0.0001) 
        
        # Simulate output tokens and content
        output_tokens = random.randint(20, max_tokens)
        response_content = "This is a mock LLM response based on your prompt. " * (output_tokens // 10)
        
        return {
            "content": response_content[:output_tokens * 4], # Approx chars per token
            "usage": {
                "prompt_tokens": input_tokens,
                "completion_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens
            }
        }

class MockTool:
    """Simulates a tool execution (e.g., database lookup, external API)."""
    def __init__(self, name: str, avg_latency: float = 0.5, avg_cost: float = 0.001):
        self.name = name
        self.avg_latency = avg_latency
        self.avg_cost = avg_cost

    def execute(self, query: str) -> dict:
        # Simulate tool execution time
        time.sleep(self.avg_latency * random.uniform(0.8, 1.2))
        
        # Simulate tool-specific cost (e.g., per query, per data retrieved)
        actual_cost = self.avg_cost * random.uniform(0.9, 1.1)
        
        return {
            "result": f"Data retrieved by {self.name} for '{query}'.",
            "cost": actual_cost
        }

class Profiler:
    """A simple profiler to track latency and cost for agentic steps."""
    def __init__(self):
        self.metrics = defaultdict(lambda: {'latency': [], 'cost': 0, 'token_usage': {'input': 0, 'output': 0}})
        self.start_time = None

    def start_session(self):
        self.start_time = time.perf_counter()

    def end_session(self, session_name="total_session"):
        if self.start_time is not None:
            end_time = time.perf_counter()
            self.metrics[session_name]['latency'].append(end_time - self.start_time)
            self.start_time = None

    def profile_llm_call(self, prompt: str, response_data: dict, step_name: str = "llm_call"):
        input_tokens = response_data['usage']['prompt_tokens']
        output_tokens = response_data['usage']['completion_tokens']
        
        llm_cost = (input_tokens / 1000 * LLM_COST_PER_1K_INPUT_TOKENS) + \
                   (output_tokens / 1000 * LLM_COST_PER_1K_OUTPUT_TOKENS)
        
        self.metrics[step_name]['cost'] += llm_cost
        self.metrics[step_name]['token_usage']['input'] += input_tokens
        self.metrics[step_name]['token_usage']['output'] += output_tokens

    def profile_tool_call(self, tool_name: str, tool_cost: float, step_latency: float):
        self.metrics[f"tool_{tool_name}"]['cost'] += tool_cost
        self.metrics[f"tool_{tool_name}"]['latency'].append(step_latency)

    def get_summary(self):
        summary = {}
        total_latency = 0
        total_cost = 0
        total_input_tokens = 0
        total_output_tokens = 0

        for step, data in self.metrics.items():
            avg_latency = sum(data['latency']) / len(data['latency']) if data['latency'] else 0
            summary[step] = {
                "avg_latency_s": f"{avg_latency:.4f}",
                "total_cost_usd": f"{data['cost']:.6f}",
                "total_input_tokens": data['token_usage']['input'],
                "total_output_tokens": data['token_usage']['output']
            }
            total_latency += sum(data['latency'])
            total_cost += data['cost']
            total_input_tokens += data['token_usage']['input']
            total_output_tokens += data['token_usage']['output']
            
        summary["overall"] = {
            "total_latency_s": f"{total_latency:.4f}",
            "total_cost_usd": f"{total_cost:.6f}",
            "total_input_tokens": total_input_tokens,
            "total_output_tokens": total_output_tokens
        }
        return summary

# --- Simulate an Agentic RAG Workflow ---

def run_agentic_rag_query(query: str, profiler: Profiler):
    print(f"\n--- Running Agent for query: '{query}' ---")
    profiler.start_session()

    # Step 1: Agent plans (LLM call)
    plan_prompt = f"Given the query '{query}', what's the best strategy to find product information?"
    start_llm_plan = time.perf_counter()
    llm_plan_response = mock_llm.generate(plan_prompt, max_tokens=50)
    end_llm_plan = time.perf_counter()
    profiler.metrics['llm_planning']['latency'].append(end_llm_plan - start_llm_plan)
    profiler.profile_llm_call(plan_prompt, llm_plan_response, 'llm_planning')
    print(f"Agent Plan: {llm_plan_response['content'][:50]}...")

    # Step 2: Agent uses a search tool (e.g., vector DB lookup)
    search_query = f"Search for products related to '{query}'"
    start_tool_search = time.perf_counter()
    search_result = mock_product_search_tool.execute(search_query)
    end_tool_search = time.perf_counter()
    profiler.profile_tool_call("product_search", search_result['cost'], end_tool_search - start_tool_search)
    print(f"Tool Search Result: {search_result['result'][:50]}...")

    # Step 3: Agent synthesizes information (LLM call)
    synthesis_prompt = f"Based on the plan '{llm_plan_response['content']}' and search results '{search_result['result']}', synthesize a response for '{query}'."
    start_llm_synthesis = time.perf_counter()
    llm_synthesis_response = mock_llm.generate(synthesis_prompt, max_tokens=150)
    end_llm_synthesis = time.perf_counter()
    profiler.metrics['llm_synthesis']['latency'].append(end_llm_synthesis - start_llm_synthesis)
    profiler.profile_llm_call(synthesis_prompt, llm_synthesis_response, 'llm_synthesis')
    print(f"Agent Synthesis: {llm_synthesis_response['content'][:50]}...")

    profiler.end_session("total_session")
    print(f"Agent Final Response: {llm_synthesis_response['content'][:100]}...")
    return llm_synthesis_response['content']

# --- Main Execution --- 

mock_llm = MockLLM()
mock_product_search_tool = MockTool(name="product_search_db", avg_latency=0.8, avg_cost=0.002)
mock_recommendation_tool = MockTool(name="recommendation_engine", avg_latency=0.3, avg_cost=0.0005)

profiler = Profiler()

queries = [
    "I need a durable laptop for coding, under $1500.",
    "Show me eco-friendly kitchen gadgets.",
    "Compare the latest smartwatches from Apple and Samsung."
]

for q in queries:
    run_agentic_rag_query(q, profiler)

print("\n--- Profiling Summary ---")
import json
print(json.dumps(profiler.get_summary(), indent=2))

# --- Example of a more complex agent path (optional, for demonstrating variability) ---
# Let's say for a specific query, the agent decides to use an additional tool
def run_complex_agentic_rag_query(query: str, profiler: Profiler):
    print(f"\n--- Running Complex Agent for query: '{query}' ---")
    profiler.start_session()

    # Step 1: Agent plans
    plan_prompt = f"Given the query '{query}', what's the best strategy to find product information?"
    start_llm_plan = time.perf_counter()
    llm_plan_response = mock_llm.generate(plan_prompt, max_tokens=50)
    end_llm_plan = time.perf_counter()
    profiler.metrics['llm_planning']['latency'].append(end_llm_plan - start_llm_plan)
    profiler.profile_llm_call(plan_prompt, llm_plan_response, 'llm_planning')
    print(f"Agent Plan: {llm_plan_response['content'][:50]}...")

    # Step 2: Agent uses a search tool
    search_query = f"Search for products related to '{query}'"
    start_tool_search = time.perf_counter()
    search_result = mock_product_search_tool.execute(search_query)
    end_tool_search = time.perf_counter()
    profiler.profile_tool_call("product_search", search_result['cost'], end_tool_search - start_tool_search)
    print(f"Tool Search Result: {search_result['result'][:50]}...")

    # Step 3: Agent decides to use a recommendation tool based on search results
    recommendation_query = f"Recommend similar items based on '{search_result['result']}'"
    start_tool_recommendation = time.perf_counter()
    recommendation_result = mock_recommendation_tool.execute(recommendation_query)
    end_tool_recommendation = time.perf_counter()
    profiler.profile_tool_call("recommendation_engine", recommendation_result['cost'], end_tool_recommendation - start_tool_recommendation)
    print(f"Tool Recommendation Result: {recommendation_result['result'][:50]}...")

    # Step 4: Agent synthesizes information from multiple sources
    synthesis_prompt = f"Based on plan '{llm_plan_response['content']}', search results '{search_result['result']}', and recommendations '{recommendation_result['result']}', synthesize a comprehensive response for '{query}'."
    start_llm_synthesis = time.perf_counter()
    llm_synthesis_response = mock_llm.generate(synthesis_prompt, max_tokens=200)
    end_llm_synthesis = time.perf_counter()
    profiler.metrics['llm_synthesis']['latency'].append(end_llm_synthesis - start_llm_synthesis)
    profiler.profile_llm_call(synthesis_prompt, llm_synthesis_response, 'llm_synthesis')
    print(f"Agent Synthesis: {llm_synthesis_response['content'][:50]}...")

    profiler.end_session("total_session")
    print(f"Agent Final Response: {llm_synthesis_response['content'][:100]}...")
    return llm_synthesis_response['content']

# Run the complex query
run_complex_agentic_rag_query("I'm looking for a high-end gaming PC, also suggest accessories.", profiler)

print("\n--- Updated Profiling Summary (after complex query) ---")
print(json.dumps(profiler.get_summary(), indent=2))


### Interpreting the Profiling Output and Performance Trade-offs

The code above simulates a simplified agentic RAG workflow and provides a basic profiling summary. Let's break down how to interpret this output and discuss the trade-offs involved.

#### Interpreting the Output

Your profiling summary will look something like this (values will vary due to simulation):

```json
{
  "llm_planning": {
    "avg_latency_s": "0.0578",
    "total_cost_usd": "0.000085",
    "total_input_tokens": 57,
    "total_output_tokens": 30
  },
  "tool_product_search_db": {
    "avg_latency_s": "0.7987",
    "total_cost_usd": "0.002000",
    "total_input_tokens": 0,
    "total_output_tokens": 0
  },
  "llm_synthesis": {
    "avg_latency_s": "0.0681",
    "total_cost_usd": "0.000180",
    "total_input_tokens": 120,
    "total_output_tokens": 60
  },
  "total_session": {
    "avg_latency_s": "0.9246",
    "total_cost_usd": "0.002265",
    "total_input_tokens": 0,
    "total_output_tokens": 0
  },
  "overall": {
    "total_latency_s": "2.7738",
    "total_cost_usd": "0.006795",
    "total_input_tokens": 531,
    "total_output_tokens": 240
  }
}
```

Here's what each part tells you:

*   **`llm_planning`**: This represents the agent's initial thought process. You'll see its average latency (how long the LLM call took), the total cost incurred by this step across all runs, and the total input/output tokens consumed. If this latency is high, it might indicate a very complex prompt or a slow LLM.
*   **`tool_product_search_db`**: This shows the performance of your product search tool. In our example, it has the highest average latency, indicating it's a potential bottleneck. Its cost is also significant, reflecting the cost of querying your vector database or external API.
*   **`llm_synthesis`**: This is where the agent formulates the final response. Its latency and token usage depend on the complexity of the information to synthesize and the desired response length.
*   **`total_session`**: This metric, when averaged, gives you the end-to-end latency for a single agentic query. It's the most direct measure of user experience.
*   **`overall`**: This provides the aggregated totals across all queries run, giving you a holistic view of the system's performance and cost over a period.

#### Performance Trade-offs and Optimization Strategies

1.  **Latency vs. Accuracy/Completeness**: A more complex agent that performs multiple tool calls or extensive reasoning might yield a more accurate or comprehensive answer, but at the cost of higher latency. For an e-commerce assistant, a quick, good-enough answer might be preferred over a perfect, slow one.
    *   **Optimization**: Implement **early exit strategies** for agents, use **smaller, faster LLMs** for initial planning or simple tasks, and **parallelize** independent agent steps or tool calls where possible.

2.  **Cost vs. Model Capability**: Larger, more capable LLMs (e.g., GPT-4o, Claude 3 Opus) are generally more expensive per token but can handle more complex tasks. Smaller models (e.g., GPT-4o mini, Llama 3 8B) are cheaper and faster but might require more careful prompting or struggle with nuanced reasoning.
    *   **Optimization**: Implement **model routing** or **cascading models**, where simpler queries go to cheaper models, and only complex ones are routed to premium models. Aggressively **summarize context** to reduce input token count.

3.  **Tool Latency**: Tools like vector database lookups or external API calls often dominate the total latency. If your `tool_product_search_db` is consistently slow, it's a major bottleneck.
    *   **Optimization**: **Cache frequently accessed data**, optimize your vector database indexing and query performance, use **serverless functions** for tools to minimize cold start times, and consider **asynchronous tool execution**.

4.  **Token Management**: Every token sent to or received from an LLM costs money. Long prompts and verbose responses add up quickly.
    *   **Optimization**: **Prompt engineering** to be concise, **response summarization** before presenting to the user, **context window compression** techniques (e.g., using smaller embedding models, re-ranking, or summarization of retrieved documents).

5.  **Observability and Tracing**: For complex agentic systems, simple `time.perf_counter()` isn't enough. Tools like **OpenTelemetry** or specialized AI observability platforms (e.g., LangSmith, Arize AI, Phoenix) provide distributed tracing, allowing you to visualize the entire agent's execution path, including all LLM calls, tool calls, and their respective latencies and costs. This is crucial for debugging and optimizing multi-agent interactions.

By continuously profiling and iterating on your agent's design, you can strike the right balance between performance, cost, and the quality of your e-commerce assistant's responses.

### Typical Use Cases for Profiling Data

*   **A/B Testing Agent Configurations**: Compare the latency and cost of different agent prompts, tool sets, or orchestration logic.
*   **Capacity Planning**: Estimate the number of LLM calls and tool invocations per second to provision adequate infrastructure.
*   **Budget Control**: Monitor actual costs against projected budgets and identify unexpected spikes.
*   **Bottleneck Identification**: Pinpoint the slowest or most expensive steps in the agent's reasoning chain.
*   **User Experience Improvement**: Directly correlate profiling data with user satisfaction metrics.
*   **Model Selection**: Inform decisions on which LLM models to use for different tasks based on their performance-to-cost ratio.


### Resources

*   **OpenTelemetry**: A vendor-neutral open-source observability framework for instrumenting, generating, collecting, and exporting telemetry data (metrics, logs, and traces). Essential for distributed tracing in complex systems.
    *   [OpenTelemetry Documentation](https://opentelemetry.io/docs/)
*   **LangChain Callbacks/Tracing**: If you're using LangChain, its callback system allows for easy integration with tracing tools and custom logging.
    *   [LangChain Callbacks](https://python.langchain.com/docs/modules/callbacks/)
    *   [LangSmith (LangChain's observability platform)](https://docs.smith.langchain.com/)
*   **AutoGen Logging**: AutoGen provides mechanisms for logging agent conversations and execution flows, which can be extended for profiling.
    *   [AutoGen Logging and Tracing](https://microsoft.github.io/autogen/docs/topics/logging_and_tracing)
*   **LLM Provider Pricing**: Understanding the pricing models of different LLMs is crucial for cost optimization.
    *   [OpenAI Pricing](https://openai.com/pricing)
    *   [Anthropic Claude Pricing](https://www.anthropic.com/api)
    *   [Google Cloud Vertex AI Pricing](https://cloud.google.com/vertex-ai/pricing)
*   **`tiktoken` Library**: For accurate token counting with OpenAI models.
    *   [GitHub: `tiktoken`](https://github.com/openai/tiktoken)
*   **AI Observability Platforms**: Dedicated platforms that provide advanced monitoring, tracing, and evaluation for AI applications.
    *   [Arize AI](https://www.arize.com/)
    *   [Phoenix (Helicone)](https://www.helicone.ai/)
    *   [Weights & Biases](https://wandb.ai/site/solutions/llm-observability)
